# MultiStyleImages · Demo GPU
Esegui le celle in ordine: configura il repository, installa le dipendenze, inizializza la demo e avvia il sito. I modelli vengono caricati solo quando premi Genera e rilasciati dopo ogni richiesta.

In Colab seleziona **Runtime → Cambia tipo di runtime → GPU**. Serve Python ≥ 3.11. Il repository completo deve essere disponibile nel runtime: puoi clonarlo o copiarlo su Drive. I checkpoint LoRA, ignorati da Git, vanno copiati separatamente. Il primo caricamento scarica diversi GB da Hugging Face.


In [ ]:
!git clone --branch demo https://github.com/LadiDBug/MultiStyleImages.git

In [ ]:
from pathlib import Path
import sys

# Modifica se il repository si trova altrove (es. /content/drive/MyDrive/MultiStyleImages).
# Su Colab, se necessario, prima monta Drive:
# from google.colab import drive
# drive.mount('/content/drive')
PROJECT_ROOT = Path('/kaggle/working/MultiStyleImages')
if not PROJECT_ROOT.is_dir():
    current = Path.cwd().resolve()
    PROJECT_ROOT = current.parent if current.name == 'demo' else current
assert (PROJECT_ROOT / 'src/generation/pipeline.py').is_file(), 'Imposta PROJECT_ROOT alla radice del repository completo.'
sys.path.insert(0, str(PROJECT_ROOT))
print('Repository:', PROJECT_ROOT)


In [ ]:
# Installa nel kernel corrente. Se aggiorna pacchetti già importati, riavvia il kernel
# e riparti dalla cella precedente prima di caricare i modelli.
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'demo/requirements.txt')])


## 1 · GPU e checkpoint LoRA
I percorsi predefiniti sono letti da `configs/lora_rank{4,8,16,32}.toml`.
`LORA_PATHS` può sostituirli, ad esempio con file su Drive. Ogni rank richiede i propri pesi addestrati: non viene addestrato alcun modello durante la demo. Senza checkpoint, Pixel art resta disponibile senza LoRA.


In [ ]:
import torch
assert torch.cuda.is_available(), 'Attiva un runtime GPU CUDA prima di continuare.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM totale:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')

LORA_PATHS = {
    4: '/kaggle/input/datasets/lachithaperera/lora-ranks/pixel_art_rank_4',
    8: '/kaggle/input/datasets/lachithaperera/lora-ranks/pixel_art_rank_8',
    16: '/kaggle/input/datasets/lachithaperera/lora-ranks/pixel_art_rank_16',
    32: '/kaggle/input/datasets/lachithaperera/lora-ranks/pixel_art_rank_32',
}


## 2 · Inizializzazione della demo
Questa cella legge le configurazioni e rileva i checkpoint LoRA, senza caricare modelli o scaricare pesi.

Quando premi **Genera**, viene estratta la sola mappa scelta. Per Depth o Segmentation viene caricato il relativo estrattore su CPU, poi eliminato prima di caricare Stable Diffusion. La generazione carica una sola pipeline: SD 1.5, il solo ControlNet selezionato (nessuno con Nessuna) e la sola LoRA scelta, se richiesta per Pixel art.

Su CUDA si usano model CPU offloading e VAE tiling. Dopo ogni generazione, anche in caso di errore, vengono rimossi gli hook, eliminati i riferimenti ai modelli ed eseguiti garbage collection e svuotamento della cache CUDA inutilizzata. I pesi restano nella cache su disco, così non devono essere scaricati a ogni richiesta. Ogni generazione richiede comunque il ricaricamento dei pesi in memoria.

Il preprocessing fotografico resta disattivato. La memoria necessaria durante l'inferenza dipende ancora dalle dimensioni della foto: immagini molto grandi possono superare la VRAM disponibile anche con una sola pipeline.


In [ ]:
from demo.backend import DemoRuntime

# Inizializzazione leggera: nessun modello viene caricato qui.
# Dopo un aggiornamento del codice, riavvia il kernel prima di rieseguire le celle.
runtime = DemoRuntime(PROJECT_ROOT, device='cuda', lora_paths=LORA_PATHS)
runtime.load()


## Dataset e risultati dei test
Configura i dizionari seguenti dopo aver caricato i dataset su Kaggle. I percorsi relativi vengono risolti rispetto a `PROJECT_ROOT`; sono ammessi anche percorsi assoluti.

`TEST_IMAGE_PATHS` punta alla cartella contenente le foto di `our_ds`. Ogni voce di `TEST_RESULT_PATHS` identifica un esperimento: baseline seed 42/123/456 oppure LoRA rank 4/8/16/32. Puoi indicare la radice dell'esportazione completa (come `outputs/rank_4`), la sua cartella `outputs`, oppure direttamente il run che contiene `samples/`. Le cartelle assenti vengono segnalate nel sito senza impedirne l'avvio.

Esempio Kaggle, se `PROJECT_ROOT` è `/kaggle/working/MultiStyleImages`: `../../input/datasets/<utente>/<dataset>/rank_4`. Sostituisci il percorso con quello effettivo del dataset. Per `our_ds` scegli la cartella delle foto, non la radice di un'esportazione contenente anche altri file.

La sezione **Esplora i test** non carica modelli e non usa la GPU. Se `our_ds` non è disponibile, può mostrare la copia `input_original.png` salvata nel test. I risultati mancanti non vengono sostituiti o generati.


In [ ]:
TEST_IMAGE_PATHS = {
    'our_ds': 'our_ds',
}

TEST_RESULT_PATHS = {
    'baseline_seed_42': 'outputs/baseline_seed_42',
    'baseline_seed_123': 'outputs/baseline_seed_123',
    'baseline_seed_456': 'outputs/baseline_seed_456',
    'rank_4': 'outputs/rank_4',
    'rank_8': 'outputs/rank_8',
    'rank_16': 'outputs/rank_16',
    'rank_32': 'outputs/rank_32',
}


## 3 · Avvio del sito
In Colab lascia `SHARE = True` e apri il link `gradio.live` stampato dalla cella. Il link è pubblico e temporaneo; puoi impostare username e password con `AUTH`. In Jupyter locale puoi usare `SHARE = False`: il sito appare nel notebook ed è disponibile all'indirizzo locale stampato.

Il kernel deve rimanere attivo. La coda esegue una generazione alla volta. Dall'immagine generata puoi scaricare il PNG; il pannello dettagli mostra la mappa e il seed utilizzati.

La demo passa la foto in RGB senza crop o resize preliminari (`use_original_image=True`). Le mappe vengono estratte dalla foto alle dimensioni originali; gli adattamenti interni sono lasciati a Diffusers.


In [ ]:
from demo.app import build_app
from demo.test_gallery import TestGallery

SHARE = True  # False per Jupyter locale senza tunnel pubblico.
AUTH = None   # Esempio: ('demo', 'scegli-una-password')
if 'app' in globals():
    app.close()
test_gallery = TestGallery(PROJECT_ROOT, image_paths=TEST_IMAGE_PATHS, result_paths=TEST_RESULT_PATHS)
app = build_app(runtime, test_gallery)
app.launch(share=SHARE, auth=AUTH, inline=True, inbrowser=False, show_error=False,
           allowed_paths=test_gallery.allowed_paths)


## 4 · Arresto del sito
I modelli vengono già rilasciati al termine di ogni generazione. La chiusura arresta il sito; per riaprirlo basta rieseguire la cella di avvio. Per passare da una vecchia versione con precaricamento a questa versione, riavvia prima il kernel e riesegui le celle.


In [ ]:
# Imposta True ed esegui questa cella soltanto quando vuoi arrestare il sito.
STOP_DEMO = False
if STOP_DEMO:
    app.close()
